# Emptyu 30-Epoch Scale Run

In [ ]:
import os, shutil, subprocess
from pathlib import Path
REPO = Path('/content/emptyu')
os.chdir('/content')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/sandeep999-cyber/emptyu.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA unavailable'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = [Path('/content/drive/MyDrive/storage'), Path('/content/drive/My Computer/storage'), Path('/content/drive/MyDrive/MarketFoundation/storage'), Path('/content/drive/My Computer/MarketFoundation/storage')]
DRIVE_STORAGE = next((p for p in CANDIDATES if p.is_dir()), None)
if DRIVE_STORAGE is None: raise FileNotFoundError('Cannot locate storage on Drive')
DRIVE_BASE = DRIVE_STORAGE.parent
src_training = DRIVE_STORAGE / 'training'
dst_training = REPO / 'storage/training'
if dst_training.exists(): shutil.rmtree(dst_training)
shutil.copytree(src_training, dst_training)
dst_canonical = REPO / 'storage/canonical'
if dst_canonical.exists(): shutil.rmtree(dst_canonical)
for symbol in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']:
    for subdir in ['klines/1m', 'funding', 'open_interest', 'metadata']:
        src = DRIVE_STORAGE / 'canonical/futures' / symbol / subdir
        dst = dst_canonical / 'futures' / symbol / subdir
        if src.is_dir(): shutil.copytree(src, dst)
for name in ['models', 'logs', 'evaluation']:
    dst, src = REPO / name, DRIVE_BASE / name
    src.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink() if dst.is_symlink() or dst.is_file() else shutil.rmtree(dst)
    dst.symlink_to(src)
import json
fp = json.loads((REPO / 'storage/training/dataset_fingerprint.json').read_text())
assert fp['fingerprint'] == '328a7b67b070b95e47ba450452032a93dfa410431e0cf329de6a4ac7b5ae3875'
assert fp['file_count'] == 510
print('Dataset verified:', fp['fingerprint'][:16] + '...')


In [ ]:
# Set RESUME_RUN to a Drive-backed run directory after a disconnect; leave None for a new run.
RESUME_RUN = None
cmd = ['python', '-u', '-m', 'src.training.train_teacher', '--model-config', 'configs/model_v1.yaml', '--optimizer-config', 'configs/optimizer_v1.yaml', '--trainer-config', 'configs/trainer_v1_scale30.yaml']
if RESUME_RUN: cmd += ['--resume', RESUME_RUN]
subprocess.run(cmd, cwd=REPO, check=True)
runs = sorted(p for p in (REPO / 'models/foundation/teacher_v1').iterdir() if (p / 'manifest.json').exists())
if not runs: raise FileNotFoundError('No completed checkpoint found')
os.environ['CHECKPOINT_DIR'] = str(runs[-1])
print('CHECKPOINT_DIR:', os.environ['CHECKPOINT_DIR'])


In [ ]:
env = dict(os.environ, PYTHONPATH=str(REPO))
subprocess.run(['python', '-u', '-m', 'src.evaluation.embedding.linear_probe', '--checkpoint', os.environ['CHECKPOINT_DIR'], '--pooling', 'all', '--batch-size', '64'], cwd=REPO, env=env, check=True)
subprocess.run(['python', '-u', '-m', 'src.evaluation.baselines.runner', '--checkpoint', os.environ['CHECKPOINT_DIR'], '--pooling', 'mean', '--max-windows', '1500', '--batch-size', '64', '--out', 'evaluation/baselines'], cwd=REPO, env=env, check=True)


In [ ]:
archive_dest = DRIVE_BASE / 'phase2_results'
shutil.make_archive(str(archive_dest), 'zip', str(REPO / 'evaluation'))
for name in ['index.duckdb', 'experiment_registry.duckdb']:
    local, remote = REPO / 'storage/training' / name, DRIVE_STORAGE / 'training' / name
    if local.exists(): shutil.copy2(local, remote)
print('Archived:', str(archive_dest) + '.zip')
